# Module 05: SOLID Principles Clean Architecture — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/clean_checkout.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import clean_checkout

classes = [n for n, o in inspect.getmembers(clean_checkout, inspect.isclass)
           if o.__module__ == 'clean_checkout']
functions = [n for n, o in inspect.getmembers(clean_checkout, inspect.isfunction)
             if o.__module__ == 'clean_checkout']

print('module   : clean_checkout')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(clean_checkout, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Money value object invariants

This is the module's own `test_money_value_object_invariants` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from decimal import Decimal

import pytest
from clean_checkout import (
    CheckoutService,
    InMemoryOrderRepository,
    MockInventoryService,
    MockPaymentGateway,
    Money,
    Order,
    OrderPaidEvent,
    OrderPlacedEvent,
    OrderStatus,
)

m1 = Money(Decimal("25.50"), "USD")
m2 = Money(Decimal("14.50"), "USD")
res = m1.add(m2)
assert res.amount == Decimal("40.00")
assert res.currency == "USD"

# Invariant 1: Negative amount rejected
with pytest.raises(ValueError, match="cannot be negative"):
    Money(Decimal("-5.00"), "USD")

# Invariant 2: Currency mismatch rejected
eur = Money(Decimal("10.00"), "EUR")
with pytest.raises(ValueError, match="different currencies"):
    m1.add(eur)

print('PASSED: test_money_value_object_invariants')

## 3. 🔮 Prediction — commit before you run

The checkout flow needs a new payment provider. Predict how many existing files must change under the clean architecture below - and what that number would be if the provider were called directly from the domain layer.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_order_aggregate_mutation_and_calculation`, which tests exactly this property.


In [ ]:
order = Order(order_id="ord-100", customer_id="cust-1", currency="USD")
order.add_item("prod-1", "Mechanical Keyboard", Money(Decimal("100.00"), "USD"), 2)
order.add_item("prod-2", "USB-C Cable", Money(Decimal("15.00"), "USD"), 3)

total = order.calculate_total()
# (100 * 2) + (15 * 3) = 245.00
assert total.amount == Decimal("245.00")
assert total.currency == "USD"

print('PASSED: test_order_aggregate_mutation_and_calculation')

## 4. Measure it: Successful checkout orchestration and events

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_successful_checkout_orchestration_and_events` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

repo = InMemoryOrderRepository()
inv = MockInventoryService(initial_stock={"prod-1": 10, "prod-2": 5})
gateway = MockPaymentGateway()
service = CheckoutService(repo, gateway, inv)

order = Order(order_id="ord-200", customer_id="cust-42", currency="USD")
order.add_item("prod-1", "Laptop Stand", Money(Decimal("45.00"), "USD"), 2)

txn_id = service.execute_checkout(order)

assert txn_id == "tx_stripe_ord-200"
assert order.status == OrderStatus.PAID
assert inv.get_available_stock("prod-1") == 8
assert repo.find_by_id("ord-200") is not None

# Check that domain events were generated
events = order.domain_events
assert len(events) == 2
assert isinstance(events[0], OrderPlacedEvent)
assert isinstance(events[1], OrderPaidEvent)
assert events[0].total_amount == Decimal("90.00")

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_successful_checkout_orchestration_and_events')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(clean_checkout) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Dependencies point inward. The domain layer knows nothing about HTTP or SQL.
2. An interface at the boundary is what makes a provider swap a one-file change.
3. SOLID is a means to changeability, not a checklist to satisfy.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
